In [1]:
!git clone https://github.com/fastapi/fastapi.git

Cloning into 'fastapi'...
remote: Enumerating objects: 79036, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 79036 (delta 30), reused 24 (delta 15), pack-reused 78967 (from 3)
Receiving objects: 100% (79036/79036), 57.64 MiB | 27.60 MiB/s, done.
Resolving deltas: 100% (59624/59624), done.


In [2]:
!pip install langchain-text-splitters langchain-community langchain-huggingface faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 MB 118.3 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-community]ngchain-community]e]


In [3]:
import glob
import os
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

all_chunks = []

md_files = glob.glob("**/docs/en/docs/**/*.md", recursive=True)
if not md_files:
    md_files = glob.glob("**/*.md", recursive=True)

print(f"Found {len(md_files)} markdown files.")

for file_path in md_files:
    if "release-notes" in file_path or "node_modules" in file_path:
        continue
        
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
    except Exception as e:
        continue
    
    splits = splitter.split_text(text)
    for doc in splits:
        if len(doc.page_content.strip()) < 100:
            continue
            
        doc.metadata["source"] = file_path
        all_chunks.append(doc)

print(f"Successfully created {len(all_chunks)} clean chunks.")

Found 158 markdown files.
Successfully created 1081 clean chunks.


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize open embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"}, # Automatically uses Kaggle's T4 GPU!
    encode_kwargs={"normalize_embeddings": True}
)

# Build the FAISS vector store
print("Embedding chunks into FAISS...")
vectorstore = FAISS.from_documents(all_chunks, embeddings)

# Save the index
vectorstore.save_local("fastapi_faiss_index")
print("Vector store successfully built and saved to 'fastapi_faiss_index'!")

# test query
query = "How do I define a Pydantic model with default values?"
results = vectorstore.similarity_search(query, k=3)

print(f"\n--- Test Search Results for: '{query}' ---")
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(doc.page_content[:300] + "...")

/tmp/ipykernel_49/600371188.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks into FAISS...
Vector store successfully built and saved to 'fastapi_faiss_index'!

--- Test Search Results for: 'How do I define a Pydantic model with default values?' ---

Result 1
Let's say you have a Pydantic model with default values, like this one:  
{* ../../docs_src/separate_openapi_schemas/tutorial001.py ln[1:7] hl[7] *}...

Result 2
First, you need to import `BaseModel` from `pydantic`:  
{* ../../docs_src/body/tutorial001.py hl[2] *}...

Result 3
You can also declare a response using a plain arbitrary `dict`, declaring just the type of the keys and values, without using a Pydantic model.  
This is useful if you don't know the valid field/attribute names (that would be needed for a Pydantic model) beforehand.  
In this case, you can use `dict...
